In [21]:
!pip install fastapi sqlalchemy uvicorn nest-asyncio pyngrok

In [22]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker, declarative_base

DATABASE_URL = "sqlite:///./test.db"

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
base = declarative_base()

In [24]:
from sqlalchemy import Column , Integer, String

class Student(Base):
    __tablename__ = 'students'
    id = Column(Integer, primary_key=True)
    name = Column(String(50))
    age = Column(Integer)
    department = Column(String(50))

In [25]:
Base.metadata.create_all(bind=engine)

In [26]:
def get_db():
  db = SessionLocal()
  try:
    yield db
  finally:
    db.close()

In [27]:
from fastapi import FastAPI , Depends
from sqlalchemy.orm import Session

app = FastAPI()

In [28]:
@app.post("/students/")
def add_student(name: str, age: int, department: str, db: Session = Depends(get_db)):
    student = Student(name=name, age=age, department=department)
    db.add(student)
    db.commit()
    db.refresh(student)
    return student

In [29]:
@app.get("/students/")
def get_students(db: Session = Depends(get_db)):
    students = db.query(Student).all()
    return students

In [30]:
@app.put("/students/{student_id}")
def update_student(student_id: int, name: str, db: Session = Depends(get_db)):
    student = db.query(Student).filter(Student.id == student_id).first()
    if student:
        student.name = name
        db.commit()
        return student
    return {"error": "Student not found"}

In [31]:
@app.delete("/students/{student_id}")
def delete_student(student_id: int, db: Session = Depends(get_db)):
    student = db.query(Student).filter(Student.id == student_id).first()
    if student:
        db.delete(student)
        db.commit()
        return {"message": "Deleted"}
    return {"error": "Student not found"}

In [32]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

uvicorn.run(app, host="0.0.0.0", port=8000)

PyngrokNgrokInstallError: An error occurred while downloading ngrok from https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.zip: HTTP Error 500: Internal Server Error

In [44]:
# Download ngrok (for Linux AMD64, which Colab uses)
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip -O ngrok.zip

--2026-03-31 04:31:43--  https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip
Resolving bin.equinox.io (bin.equinox.io)... 35.71.179.82, 75.2.60.68, 13.248.244.96, ...
Connecting to bin.equinox.io (bin.equinox.io)|35.71.179.82|:443... connected.
HTTP request sent, awaiting response... 500 Internal Server Error
2026-03-31 04:31:43 ERROR 500: Internal Server Error.



In [45]:
# Unzip the downloaded file
!unzip -o ngrok.zip

Archive:  ngrok.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of ngrok.zip or
        ngrok.zip.zip, and cannot find ngrok.zip.ZIP, period.


In [46]:
# Move the ngrok executable to /usr/local/bin to make it accessible
!mv ngrok /usr/local/bin/

mv: cannot stat 'ngrok': No such file or directory


In [47]:
# Verify ngrok installation
!ngrok version

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pyngrok/installer.py", line 190, in install_ngrok
    download_path = _download_file(url, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pyngrok/installer.py", line 385, in _download_file
    raise e
  File "/usr/local/lib/python3.12/dist-packages/pyngrok/installer.py", line 343, in _download_file
    response = urlopen(url, **kwargs)
               ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 215, in urlopen
    return opener.open(url, data, timeout)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 521, in open
    response = meth(req, response)
               ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py", line 630, in http_response
    response = self.parent.error(
               ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/urllib/request.py",

In [48]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# Set the ngrok_path to the manually installed ngrok executable
from pyngrok import conf
conf.get_default().ngrok_path = "/usr/local/bin/ngrok"

public_url = ngrok.connect(8000)
print("Public URL:", public_url)

uvicorn.run(app, host="0.0.0.0", port=8000)

PyngrokNgrokError: The ngrok process was unable to start.

In [ ]:
# Create the .cloudflared directory if it doesn't exist
!mkdir -p ~/.cloudflared

# Move the downloaded cert.pem to the correct location
# Make sure you have uploaded the cert.pem file to the current working directory in Colab first.
!mv cert.pem ~/.cloudflared/

Once the `cert.pem` file is in place, we can create and run the Cloudflare Tunnel. First, you'll need to choose a name for your tunnel. Then, we will create the tunnel and a `config.yml` file.

In [ ]:
import subprocess

tunnel_name = "my-fastapi-tunnel" # You can choose any name for your tunnel

# Create a tunnel (this will also create a credential file in ~/.cloudflared/)
result = subprocess.run(['./cloudflared', 'tunnel', 'create', tunnel_name], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

# Extract the tunnel ID from the output
import re
match = re.search(r'ID: ([a-f0-9-]+)', result.stdout)
if match:
    tunnel_id = match.group(1)
    print(f"Tunnel ID: {tunnel_id}")
else:
    print("Could not find Tunnel ID in the output.")
    tunnel_id = None

# Create a config.yml file for the tunnel
if tunnel_id:
    config_content = f"""
url: http://localhost:8000
tunnel: {tunnel_id}
credentials-file: /root/.cloudflared/{tunnel_id}.json
"""
    with open("config.yml", "w") as f:
        f.write(config_content)
    print("config.yml created.")
else:
    print("Cannot create config.yml without a Tunnel ID.")

In [49]:
import requests
import os

# Download the cloudflared binary for Linux AMD64 (Colab environment)
url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors

with open("cloudflared", "wb") as f:
    f.write(response.content)

# Make the binary executable
os.chmod("cloudflared", 0o755)

print("cloudflared downloaded and made executable.")

cloudflared downloaded and made executable.


Next, you'll need to authenticate `cloudflared` with your Cloudflare account. This requires you to have a Cloudflare account and a domain associated with it. Please run the following command and follow the instructions to log in through your browser. This will generate a `cert.pem` file that `cloudflared` uses for authentication.

In [50]:
!./cloudflared tunnel login

A browser window should have opened at the following URL:

https://dash.cloudflare.com/argotunnel?aud=&callback=https%3A%2F%2Flogin.cloudflareaccess.org%2FnIPOiBCkeYQMeoEbVd5kix72PBMBkiMUJR5UvbRoSWc%3D

If the browser failed to open, please visit the URL above directly in your browser.
2026-03-31T04:33:40Z INF Waiting for login...
2026-03-31T04:34:33Z INF Waiting for login...
2026-03-31T04:35:26Z INF Waiting for login...
2026-03-31T04:36:19Z INF Waiting for login...
2026-03-31T04:37:12Z INF Waiting for login...
2026-03-31T04:38:04Z INF Waiting for login...
2026-03-31T04:38:57Z INF Waiting for login...
2026-03-31T04:39:50Z INF Waiting for login...
2026-03-31T04:40:42Z INF Waiting for login...
2026-03-31T04:41:35Z INF Waiting for login...
2026-03-31T04:41:35Z ERR Failed to write the certificate.

Your browser will download the certificate instead. You will have to manually
copy it to the following path:

/root/.cloudflared/cert.pem
 error="Failed to fetch resource"
Failed to fetch resour